# SARIMA — demanda y precio de combustible

Búsqueda de órdenes por AIC, ajuste SARIMA, diagnóstico de residuos y pronóstico
semanal con intervalo de confianza al 95 %, más ruido gaussiano para escalar el
pronóstico a cada estación.

Los archivos `.xlsx` de entrada no se publican (datos del cliente); las fuentes
públicas son SENER (demanda) y CRE (precios).


codigo adaptado a datos de gasolineras. Empieza primero buscando los coeficientes. La busqueda de coeficientes se hace por medio de AIC, Criterio de Información de Akaike.
## AIC para modelos (incluido SARIMA)

### 1. Definición del AIC

Para cualquier modelo (incluido SARIMA), el AIC se define como:

$$
\text{AIC} = -2 \ln(L) + 2k
$$

donde:

- \(L\) = máxima verosimilitud del modelo (qué tan bien se ajusta a los datos)
- \(k\) = número de parámetros estimados del modelo

**Idea:**

- Cuanto mejor se ajuste el modelo → más grande es \(L\) → más pequeño es \(-2\ln(L)\).
- Cuantos más parámetros tenga el modelo → más grande es \(2k\) → el AIC sube (penaliza la complejidad).

**Conclusión:** un buen modelo (entre varios candidatos) es el que tiene el **AIC más bajo**.

---

### 2. AIC específicamente en modelos SARIMA

Un modelo SARIMA se suele escribir como:

$$
\text{SARIMA}(p,d,q)(P,D,Q)_s
$$

donde:

- **Parte no estacional**:
  - \(p\): número de términos autorregresivos (AR)
  - \(d\): orden de diferenciación
  - \(q\): número de términos de medias móviles (MA)

- **Parte estacional**:
  - \(P\): términos AR estacionales
  - \(D\): orden de diferenciación estacional
  - \(Q\): términos MA estacionales
  - \(s\): período de estacionalidad (por ejemplo, \(s = 12\) si hay patrón anual en datos mensuales)

Cada parámetro AR/MA (y, según el modelo, la constante, varianza del error, etc.) aporta al conteo \(k\).

Cuando ajustas varios modelos SARIMA (probando diferentes \(p,d,q,P,D,Q\)), **cada uno genera un AIC distinto**.

**Ejemplo de comparación:**

- $\text{SARIMA}(1,1,1)(0,1,1)_{12}$ → AIC = 1020  
- $\text{SARIMA}(0,1,1)(0,1,1)_{12}$ → AIC = 1010  
- $\text{SARIMA}(2,1,1)(1,1,1)_{12}$ → AIC = 1008  

El modelo preferido sería el de **AIC = 1008**, porque es el **más bajo**.

---

### 3. Cómo interpretar diferencias de AIC

Solo tiene sentido comparar AIC entre **modelos ajustados a la misma serie** y al **mismo rango de datos**.

Una regla típica:

- $\Delta \text{AIC} < 2$ → modelos prácticamente igual de buenos  
- $4 \leq \Delta \text{AIC} < 7$ → evidencia moderada a favor del de menor AIC  
- $\Delta \text{AIC} \geq 10$ → el modelo con mayor AIC es claramente peor

donde:

$$
\Delta \text{AIC} = \text{AIC}_{\text{modelo}} - \text{AIC}_{\text{mejor modelo}}
$$

---

### 4. Cosas importantes a recordar

- El AIC **no dice si un modelo es “bueno” en términos absolutos**, solo cuál es **mejor** dentro del conjunto de modelos que estás comparando.
- Penaliza la complejidad del modelo, evitando que elijas un SARIMA con demasiados parámetros solo porque ajusta muy bien los datos de entrenamiento (*overfitting*).
- En la práctica, además del AIC, se revisa:
  - Diagnóstico de residuos (autocorrelación, normalidad, varianza constante, etc.).
  - Desempeño en pronósticos (por ejemplo, error en un conjunto de validación).

---
 ejemplo de un aic simplificado

In [ ]:
# ============================
# SARIMA
# - Busca órdenes por AIC
# - Extrae coeficientes
# - Grafica proyección
# ============================

import pandas as pd
import numpy as np
import warnings
import matplotlib.pyplot as plt
import statsmodels.api as sm
from statsmodels.tools.sm_exceptions import ConvergenceWarning, ValueWarning as SMValueWarning

# 1. Leer datos desde Excel --------------------------
ruta_archivo = "datos sarima.xlsx"  # cambia la ruta si está en otra carpeta
df = pd.read_excel(ruta_archivo)


# 2. Función: reconstruir serie semanal de gasolina --
def construir_serie_gasolina(df: pd.DataFrame) -> pd.Series:
    """
    Toma tu archivo 'datos sarima.xlsx' y devuelve una serie semanal
    con el precio de gasolina en Nuevo León (todas las semanas 2022–2025).
    """

    # Bloque de columnas donde está la tabla de gasolina
    # (es el bloque que empieza en "Gasolina en Nuevo León")
    bloque = df.iloc[1:, 8:15].copy()  # columnas 8–14 (0-based)
    header = bloque.iloc[0]            # fila que contiene: semana_numero, Mes, 2022, 2023, 2024, 2025
    data = bloque.iloc[1:].copy()      # resto de filas son datos

    # Ponemos nombres de columna correctos
    data.columns = header  # ahora: ['semana_numero', 'Mes', 2022.0, 2023.0, 2024.0, 2025.0, NaN]

    # Nos quedamos con columnas útiles
    data = data[['semana_numero', 'Mes', 2022.0, 2023.0, 2024.0, 2025.0]]
    data = data[data['semana_numero'].notna()]
    data['semana_numero'] = data['semana_numero'].astype(int)

    # Pasamos años a formato "largo": una fila por (año, semana)
    value_cols = [2022.0, 2023.0, 2024.0, 2025.0]
    largo = data.melt(
        id_vars=['semana_numero', 'Mes'],
        value_vars=value_cols,
        var_name='year',
        value_name='precio'
    )

    # Quitamos filas sin dato
    largo = largo.dropna(subset=['precio'])
    largo['year'] = largo['year'].astype(int)

    # Creamos una fecha: lunes de cada semana ISO (año + número de semana)
    largo['fecha'] = pd.to_datetime(
        largo['year'].astype(str)
        + '-W'
        + largo['semana_numero'].astype(int).astype(str)
        + '-1',           # 1 = lunes
        format='%G-W%V-%u'
    )

    # Ordenamos por fecha y construimos la serie
    largo = largo.sort_values('fecha')
    y = largo.set_index('fecha')['precio'].asfreq('W-MON')  # frecuencia semanal (lunes)
    y = y.astype(float)

    return y


y = construir_serie_gasolina(df)
print("Serie construida con", len(y), "observaciones.")
print("Desde:", y.index.min().date(), "hasta:", y.index.max().date())


# 3. Búsqueda de mejor SARIMA por AIC ----------------

# Estacionalidad semanal con patrón anual (52 semanas)
M = 52

# Rango moderado de órdenes
p_range = [0, 1, 2]
d_range = [0, 1]
q_range = [0, 1, 2]
P_range = [0, 1]
D_range = [0, 1]
Q_range = [0, 1]

best_aic = np.inf
best_order = None
best_seasonal = None
best_res = None
total_tries = 0

for p in p_range:
    for d in d_range:
        for q in q_range:
            for P in P_range:
                for D in D_range:
                    for Q in Q_range:
                        # Evitar (0,0,0)x(0,0,0)
                        if (p, d, q) == (0, 0, 0) and (P, D, Q) == (0, 0, 0):
                            continue

                        order = (p, d, q)
                        seasonal_order = (P, D, Q, M)
                        total_tries += 1

                        try:
                            modelo = sm.tsa.statespace.SARIMAX(
                                y,
                                order=order,
                                seasonal_order=seasonal_order,
                                trend="n",
                                enforce_stationarity=False,
                                enforce_invertibility=False,
                            )

                            # Convertimos ciertos warnings en errores solo durante el ajuste
                            with warnings.catch_warnings():
                                warnings.filterwarnings("error", category=ConvergenceWarning)
                                warnings.filterwarnings(
                                    "error",
                                    category=SMValueWarning,
                                    module=r"statsmodels\.tsa\.base\.tsa_model"
                                )
                                res = modelo.fit(disp=False)

                            if res.aic < best_aic:
                                best_aic = res.aic
                                best_order = order
                                best_seasonal = seasonal_order
                                best_res = res

                        except (ConvergenceWarning, SMValueWarning):
                            # Modelos problemáticos (no convergen, etc.)
                            continue
                        except Exception:
                            # Otros errores numéricos, los ignoramos
                            pass

print(f"\nIntentos evaluados: {total_tries}")
if best_res is None:
    raise RuntimeError("No se pudo ajustar ningún modelo en la rejilla.")

print(">>> Mejor modelo por AIC:")
print("order (p,d,q)           :", best_order)
print("seasonal_order (P,D,Q,s):", best_seasonal)
print("AIC:", best_aic)


# 4. Extraer coeficientes SARIMA ---------------------

params = dict(zip(best_res.param_names, best_res.params))

phi, theta, Phi, Theta = [], [], [], []

for nombre, valor in params.items():
    # AR / MA no estacional
    if nombre.startswith("ar.L"):
        phi.append(float(valor))
    elif nombre.startswith("ma.L"):
        theta.append(float(valor))
    # AR / MA estacional
    elif nombre.startswith("ar.S.L"):
        Phi.append(float(valor))
    elif nombre.startswith("ma.S.L"):
        Theta.append(float(valor))

# Recortar según órdenes del mejor modelo
p, d, q = best_order
P, D, Q, s = best_seasonal

phi   = phi[:p]
theta = theta[:q]
Phi   = Phi[:P]
Theta = Theta[:Q]

# Desviación estándar del error
sigma_eps = float(np.sqrt(params.get("sigma2", np.nan)))

print("\n>>> Coeficientes del mejor SARIMA:")
print("phi   (AR no estacional):", phi)
print("theta (MA no estacional):", theta)
print("Phi   (AR estacional)   :", Phi)
print("Theta (MA estacional)   :", Theta)
print("sigma_eps (ruido)       :", sigma_eps)


# 5. Proyección y gráfica ----------------------------

pasos_forecast = 20  # nº de semanas hacia adelante
forecast = best_res.get_forecast(steps=pasos_forecast)

pred_media = forecast.predicted_mean
conf_int = forecast.conf_int()  # intervalos de confianza (por columna)

plt.figure(figsize=(12, 5))

# Serie histórica
plt.plot(y.index, y, label="Histórico", linewidth=2)

# Pronóstico
plt.plot(pred_media.index, pred_media, label="Pronóstico", linestyle="--")

# Banda de confianza
plt.fill_between(
    pred_media.index,
    conf_int.iloc[:, 0],
    conf_int.iloc[:, 1],
    alpha=0.2,
    label="IC 95%"
)

plt.title("Gasolina en Nuevo León – Modelo SARIMA\nHistórico y proyección semanal")
plt.xlabel("Fecha")
plt.ylabel("Precio")
plt.grid(True)
plt.legend()
plt.tight_layout()
plt.show()


In [ ]:
# ============================
# SARIMA
# - Busca órdenes por AIC
# - Extrae coeficientes
# - Grafica proyección
# ============================

import pandas as pd
import numpy as np
import warnings
import matplotlib.pyplot as plt
import statsmodels.api as sm
from statsmodels.tools.sm_exceptions import ConvergenceWarning, ValueWarning as SMValueWarning

# 1. Leer datos desde Excel --------------------------
ruta_archivo = "datos sarima.xlsx"  # cambia la ruta si está en otra carpeta
df = pd.read_excel(ruta_archivo)


# 2. Función: reconstruir serie semanal de gasolina --
def construir_serie_gasolina(df: pd.DataFrame) -> pd.Series:
    """
    Toma tu archivo 'datos sarima.xlsx' y devuelve una serie semanal
    con el precio de gasolina en Nuevo León (todas las semanas 2022–2025).
    """

    # Bloque de columnas donde está la tabla de DIESEL
    # (es el bloque que empieza en "Gasolina en Nuevo León")
    bloque = df.iloc[1:, 15:21].copy()  # columnas 8–14 (0-based)
    header = bloque.iloc[0]            # fila que contiene: semana_numero, Mes, 2022, 2023, 2024, 2025
    data = bloque.iloc[1:].copy()      # resto de filas son datos

    # Ponemos nombres de columna correctos
    data.columns = header  # ahora: ['semana_numero', 'Mes', 2022.0, 2023.0, 2024.0, 2025.0, NaN]

    # Nos quedamos con columnas útiles
    data = data[['semana_numero', 'Mes', 2022.0, 2023.0, 2024.0, 2025.0]]
    data = data[data['semana_numero'].notna()]
    data['semana_numero'] = data['semana_numero'].astype(int)

    # Pasamos años a formato "largo": una fila por (año, semana)
    value_cols = [2022.0, 2023.0, 2024.0, 2025.0]
    largo = data.melt(
        id_vars=['semana_numero', 'Mes'],
        value_vars=value_cols,
        var_name='year',
        value_name='precio'
    )

    # Quitamos filas sin dato
    largo = largo.dropna(subset=['precio'])
    largo['year'] = largo['year'].astype(int)

    # Creamos una fecha: lunes de cada semana ISO (año + número de semana)
    largo['fecha'] = pd.to_datetime(
        largo['year'].astype(str)
        + '-W'
        + largo['semana_numero'].astype(int).astype(str)
        + '-1',           # 1 = lunes
        format='%G-W%V-%u'
    )

    # Ordenamos por fecha y construimos la serie
    largo = largo.sort_values('fecha')
    y = largo.set_index('fecha')['precio'].asfreq('W-MON')  # frecuencia semanal (lunes)
    y = y.astype(float)

    return y


y = construir_serie_gasolina(df)
print("Serie construida con", len(y), "observaciones.")
print("Desde:", y.index.min().date(), "hasta:", y.index.max().date())


# 3. Búsqueda de mejor SARIMA por AIC ----------------

# Estacionalidad semanal con patrón anual (52 semanas)
M = 52

# Rango moderado de órdenes
p_range = [0, 1, 2]
d_range = [0, 1]
q_range = [0, 1, 2]
P_range = [0, 1]
D_range = [0, 1]
Q_range = [0, 1]

best_aic = np.inf
best_order = None
best_seasonal = None
best_res = None
total_tries = 0

for p in p_range:
    for d in d_range:
        for q in q_range:
            for P in P_range:
                for D in D_range:
                    for Q in Q_range:
                        # Evitar (0,0,0)x(0,0,0)
                        if (p, d, q) == (0, 0, 0) and (P, D, Q) == (0, 0, 0):
                            continue

                        order = (p, d, q)
                        seasonal_order = (P, D, Q, M)
                        total_tries += 1

                        try:
                            modelo = sm.tsa.statespace.SARIMAX(
                                y,
                                order=order,
                                seasonal_order=seasonal_order,
                                trend="n",
                                enforce_stationarity=False,
                                enforce_invertibility=False,
                            )

                            # Convertimos ciertos warnings en errores solo durante el ajuste
                            with warnings.catch_warnings():
                                warnings.filterwarnings("error", category=ConvergenceWarning)
                                warnings.filterwarnings(
                                    "error",
                                    category=SMValueWarning,
                                    module=r"statsmodels\.tsa\.base\.tsa_model"
                                )
                                res = modelo.fit(disp=False)

                            if res.aic < best_aic:
                                best_aic = res.aic
                                best_order = order
                                best_seasonal = seasonal_order
                                best_res = res

                        except (ConvergenceWarning, SMValueWarning):
                            # Modelos problemáticos (no convergen, etc.)
                            continue
                        except Exception:
                            # Otros errores numéricos, los ignoramos
                            pass

print(f"\nIntentos evaluados: {total_tries}")
if best_res is None:
    raise RuntimeError("No se pudo ajustar ningún modelo en la rejilla.")

print(">>> Mejor modelo por AIC:")
print("order (p,d,q)           :", best_order)
print("seasonal_order (P,D,Q,s):", best_seasonal)
print("AIC:", best_aic)


# 4. Extraer coeficientes SARIMA ---------------------

params = dict(zip(best_res.param_names, best_res.params))

phi, theta, Phi, Theta = [], [], [], []

for nombre, valor in params.items():
    # AR / MA no estacional
    if nombre.startswith("ar.L"):
        phi.append(float(valor))
    elif nombre.startswith("ma.L"):
        theta.append(float(valor))
    # AR / MA estacional
    elif nombre.startswith("ar.S.L"):
        Phi.append(float(valor))
    elif nombre.startswith("ma.S.L"):
        Theta.append(float(valor))

# Recortar según órdenes del mejor modelo
p, d, q = best_order
P, D, Q, s = best_seasonal

phi   = phi[:p]
theta = theta[:q]
Phi   = Phi[:P]
Theta = Theta[:Q]

# Desviación estándar del error
sigma_eps = float(np.sqrt(params.get("sigma2", np.nan)))

print("\n>>> Coeficientes del mejor SARIMA:")
print("phi   (AR no estacional):", phi)
print("theta (MA no estacional):", theta)
print("Phi   (AR estacional)   :", Phi)
print("Theta (MA estacional)   :", Theta)
print("sigma_eps (ruido)       :", sigma_eps)


# 5. Proyección y gráfica ----------------------------

pasos_forecast = 20  # nº de semanas hacia adelante
forecast = best_res.get_forecast(steps=pasos_forecast)

pred_media = forecast.predicted_mean
conf_int = forecast.conf_int()  # intervalos de confianza (por columna)

plt.figure(figsize=(12, 5))

# Serie histórica
plt.plot(y.index, y, label="Histórico", linewidth=2)

# Pronóstico
plt.plot(pred_media.index, pred_media, label="Pronóstico", linestyle="--")

# Banda de confianza
plt.fill_between(
    pred_media.index,
    conf_int.iloc[:, 0],
    conf_int.iloc[:, 1],
    alpha=0.2,
    label="IC 95%"
)

plt.title("Gasolina en Nuevo León – Modelo SARIMA\nHistórico y proyección semanal")
plt.xlabel("Fecha")
plt.ylabel("Precio")
plt.grid(True)
plt.legend()
plt.tight_layout()
plt.show()


Graficas de gasolina regular por estacion

In [ ]:
import pandas as pd
import numpy as np
import statsmodels.api as sm
import matplotlib.pyplot as plt


phi   = []  # AR no estacional
theta = [-0.9487605332773176, 0.15680455524202244]  # MA no estacional
Phi   = []  # AR estacional
Theta = [-0.2096102495916201]  # MA estacional
sigma_eps = 2.0955401139243035  # Ruido

# 1. Leer datos desde Excel --------------------------
ruta_archivo = "datos sarima.xlsx"  # Cambiar ruta si es necesario
df = pd.read_excel(ruta_archivo)

def construir_serie_gasolina(df: pd.DataFrame) -> pd.Series:
    """
    Función para construir la serie semanal de precios de gasolina.
    """
    bloque = df.iloc[1:, 8:14].copy()
    header = bloque.iloc[0]  # Fila que contiene semana_numero, Mes, 2022, 2023, 2024, 2025
    data = bloque.iloc[1:].copy()

    data.columns = header  # Renombramos las columnas
    data = data[['semana_numero', 'Mes', 2022.0, 2023.0, 2024.0, 2025.0]]
    data = data[data['semana_numero'].notna()]
    data['semana_numero'] = data['semana_numero'].astype(int)

    # Convertir a formato largo: una fila por año y semana
    value_cols = [2022.0, 2023.0, 2024.0, 2025.0]
    largo = data.melt(
        id_vars=['semana_numero', 'Mes'],
        value_vars=value_cols,
        var_name='year',
        value_name='precio'
    )

    largo = largo.dropna(subset=['precio'])
    largo['year'] = largo['year'].astype(int)

    # Crear la fecha: lunes de cada semana ISO (año + número de semana)
    largo['fecha'] = pd.to_datetime(
        largo['year'].astype(str)
        + '-W'
        + largo['semana_numero'].astype(int).astype(str)
        + '-1',  # 1 = lunes
        format='%G-W%V-%u'
    )

    largo = largo.sort_values('fecha')
    y = largo.set_index('fecha')['precio'].asfreq('W-MON')  # Frecuencia semanal (lunes)
    y = y.astype(float)

    return y

# Construir la serie
y = construir_serie_gasolina(df)

# 2. Ajustar el modelo SARIMA con los coeficientes dados
order = (0, 1, 2)  # p, d, q
seasonal_order = (1, 1, 1, 52)  # P, D, Q, s (estacionalidad semanal con 52 semanas)

# Crear y ajustar el modelo SARIMA
modelo = sm.tsa.statespace.SARIMAX(
    y,
    order=order,
    seasonal_order=seasonal_order,
    trend="n",
    enforce_stationarity=False,
    enforce_invertibility=False,
    start_params=[*phi, *theta, *Phi, *Theta, sigma_eps]
)

# Ajustar el modelo SARIMA
res = modelo.fit(disp=False)

# Resumen del modelo
print(res.summary())

# 3. Proyección y gráfica ----------------------------
pasos_forecast = 30  # Número de semanas hacia adelante
forecast = res.get_forecast(steps=pasos_forecast)

pred_media = forecast.predicted_mean
conf_int = forecast.conf_int()  # Intervalos de confianza (por columna)

a = [166.3359, 75.51331, 62.09946, 42.6542, 74.7195]  # Escaladores para cada gráfico
proyecciones_con_ruido = []  # Lista para almacenar las proyecciones con ruido

# Bucle para generar las gráficas con ruido
n_estaciones = 5

proyecciones_con_ruido = []
historicos_con_ruido = []

for i in range(n_estaciones):
    # Añadir ruido gaussiano a la serie histórica
    ruido_gaussiano = np.random.normal(0, sigma_eps, size=len(y))  # Ruido con media 0 y desviación sigma_eps
    y_con_ruido = y * a[i] + ruido_gaussiano  # Agregar ruido a la serie histórica

    # Guardar histórico con ruido de esta estación
    historicos_con_ruido.append(y_con_ruido)

    # Generamos ruido para las proyecciones (predicciones) también
    ruido_gaussiano2 = np.random.normal(0, sigma_eps, size=len(pred_media))
    proyeccion_con_ruido = pred_media * a[i] + ruido_gaussiano2 * 0.8 * a[i]  # Ajuste con ruido adicional

    # Guardar proyección con ruido de esta estación
    proyecciones_con_ruido.append(proyeccion_con_ruido)

    # Gráfica de la serie histórica y la proyección
    plt.figure(figsize=(12, 5))

    # Serie histórica con ruido
    plt.plot(y.index, y_con_ruido, label="Histórico (con ruido)", linewidth=2)

    # Pronóstico con ruido
    plt.plot(pred_media.index, proyeccion_con_ruido, label="Pronóstico (con ruido)", linestyle="--")

    # Banda de confianza (escalada por el factor de la estación)
    plt.fill_between(
        pred_media.index,
        conf_int.iloc[:, 0] * a[i],
        conf_int.iloc[:, 1] * a[i],
        alpha=0.2,
        label="IC 95%"
    )

    plt.title(f"Proyección de estación {i + 1}")
    plt.xlabel("Fecha")
    plt.ylabel("Litros por día")
    plt.grid(True)
    plt.legend()
    plt.tight_layout()
    plt.show()

# -------------------------------
# DataFrames con estaciones como columnas
# -------------------------------

# Históricos con ruido (filas = fechas históricas, columnas = estaciones)
df_historico = pd.DataFrame(
    {f'Estacion_{i+1}': historicos_con_ruido[i] for i in range(n_estaciones)},
    index=y.index
)

# Proyecciones con ruido (filas = fechas de predicción, columnas = estaciones)
df_proyecciones = pd.DataFrame(
    {f'Estacion_{i+1}': proyecciones_con_ruido[i] for i in range(n_estaciones)},
    index=pred_media.index
)

# -------------------------------
# Exportar a Excel
# -------------------------------
with pd.ExcelWriter("resultados_estaciones.xlsx") as writer:
    df_historico.to_excel(writer, sheet_name="Historico")
    df_proyecciones.to_excel(writer, sheet_name="Proyecciones")

Graficas de gasolina premiun por estacion

**Predicción precio de la gasolina**

In [ ]:
# ============================
# SARIMA
# - Busca órdenes por AIC
# - Extrae coeficientes
# - Grafica proyección
# ============================

import pandas as pd
import numpy as np
import warnings
import matplotlib.pyplot as plt
import statsmodels.api as sm
from statsmodels.tools.sm_exceptions import ConvergenceWarning, ValueWarning as SMValueWarning

# 1. Leer datos desde Excel --------------------------
ruta_archivo = "PrecioGasolina.xlsx"  # cambia la ruta si está en otra carpeta
df = pd.read_excel(ruta_archivo)


def construir_serie_gasolina(df: pd.DataFrame) -> pd.Series:
    """
    A partir de un DataFrame con columnas:
        - 'Dia' (1 = 1/ene/2017)
        - 'GasolinaRegular'
    Devuelve una serie semanal (lunes) con los últimos 4 años de datos.
    """

    # 1. Convertir "Dia" → fecha real
    fecha_inicio = pd.Timestamp("2017-01-01")
    df = df.copy()
    df["fecha"] = fecha_inicio + pd.to_timedelta(df["Dia"] - 1, unit="D")

    # 2. Ordenar por fecha por si viene desordenado
    df = df.sort_values("fecha")

    # 3. Quedarnos solo con los últimos 4 años de datos
    fecha_max = df["fecha"].max()
    fecha_corte = fecha_max - pd.DateOffset(years=4)
    df = df[df["fecha"] >= fecha_corte]

    # 4. Crear serie diaria
    serie_diaria = df.set_index("fecha")["GasolinaRegular"].astype(float)

    # 5. Convertir a serie semanal (lunes)
    #    Método: tomar el promedio semanal o el valor del lunes.
    #    Aquí uso .resample('W-MON').mean()
    serie_semanal = serie_diaria.resample("W-MON").mean()

    return serie_semanal


y = construir_serie_gasolina(df)
print("Serie construida con", len(y), "observaciones.")
print("Desde:", y.index.min().date(), "hasta:", y.index.max().date())


# 3. Búsqueda de mejor SARIMA por AIC ----------------

# Estacionalidad semanal con patrón anual (208 semanas)
M = 208

# Rango moderado de órdenes
p_range = [0, 1, 2]
d_range = [0, 1]
q_range = [0, 1, 2]
P_range = [0, 1]
D_range = [0, 1]
Q_range = [0, 1]

best_aic = np.inf
best_order = None
best_seasonal = None
best_res = None
total_tries = 0

for p in p_range:
    for d in d_range:
        for q in q_range:
            for P in P_range:
                for D in D_range:
                    for Q in Q_range:
                        # Evitar (0,0,0)x(0,0,0)
                        if (p, d, q) == (0, 0, 0) and (P, D, Q) == (0, 0, 0):
                            continue

                        order = (p, d, q)
                        seasonal_order = (P, D, Q, M)
                        total_tries += 1

                        try:
                            modelo = sm.tsa.statespace.SARIMAX(
                                y,
                                order=order,
                                seasonal_order=seasonal_order,
                                trend="n",
                                enforce_stationarity=False,
                                enforce_invertibility=False,
                            )

                            # Convertimos ciertos warnings en errores solo durante el ajuste
                            with warnings.catch_warnings():
                                warnings.filterwarnings("error", category=ConvergenceWarning)
                                warnings.filterwarnings(
                                    "error",
                                    category=SMValueWarning,
                                    module=r"statsmodels\.tsa\.base\.tsa_model"
                                )
                                res = modelo.fit(disp=False)

                            if res.aic < best_aic:
                                best_aic = res.aic
                                best_order = order
                                best_seasonal = seasonal_order
                                best_res = res

                        except (ConvergenceWarning, SMValueWarning):
                            # Modelos problemáticos (no convergen, etc.)
                            continue
                        except Exception:
                            # Otros errores numéricos, los ignoramos
                            pass

print(f"\nIntentos evaluados: {total_tries}")
if best_res is None:
    raise RuntimeError("No se pudo ajustar ningún modelo en la rejilla.")

print(">>> Mejor modelo por AIC:")
print("order (p,d,q)           :", best_order)
print("seasonal_order (P,D,Q,s):", best_seasonal)
print("AIC:", best_aic)


# 4. Extraer coeficientes SARIMA ---------------------

params = dict(zip(best_res.param_names, best_res.params))

phi, theta, Phi, Theta = [], [], [], []

for nombre, valor in params.items():
    # AR / MA no estacional
    if nombre.startswith("ar.L"):
        phi.append(float(valor))
    elif nombre.startswith("ma.L"):
        theta.append(float(valor))
    # AR / MA estacional
    elif nombre.startswith("ar.S.L"):
        Phi.append(float(valor))
    elif nombre.startswith("ma.S.L"):
        Theta.append(float(valor))

# Recortar según órdenes del mejor modelo
p, d, q = best_order
P, D, Q, s = best_seasonal

phi   = phi[:p]
theta = theta[:q]
Phi   = Phi[:P]
Theta = Theta[:Q]

# Desviación estándar del error
sigma_eps = float(np.sqrt(params.get("sigma2", np.nan)))

print("\n>>> Coeficientes del mejor SARIMA:")
print("phi   (AR no estacional):", phi)
print("theta (MA no estacional):", theta)
print("Phi   (AR estacional)   :", Phi)
print("Theta (MA estacional)   :", Theta)
print("sigma_eps (ruido)       :", sigma_eps)


# 5. Proyección y gráfica ----------------------------

pasos_forecast = 20  # nº de semanas hacia adelante
forecast = best_res.get_forecast(steps=pasos_forecast)

pred_media = forecast.predicted_mean
conf_int = forecast.conf_int()  # intervalos de confianza (por columna)

plt.figure(figsize=(12, 5))

# Serie histórica
plt.plot(y.index, y, label="Histórico", linewidth=2)

# Pronóstico
plt.plot(pred_media.index, pred_media, label="Pronóstico", linestyle="--")

# Banda de confianza
plt.fill_between(
    pred_media.index,
    conf_int.iloc[:, 0],
    conf_int.iloc[:, 1],
    alpha=0.2,
    label="IC 95%"
)

plt.title("Precio de Gasolina en México – Modelo SARIMA\nHistórico y proyección semanal")
plt.xlabel("Fecha")
plt.ylabel("Precio [Pesos/Litro]")
plt.grid(True)
plt.legend()
plt.tight_layout()
plt.show()


In [ ]:
import pandas as pd
import numpy as np
import statsmodels.api as sm
import matplotlib.pyplot as plt


phi   = [1.349808889224366, -0.34935220796663913]  # AR no estacional
theta = []  # MA no estacional
Phi   = []  # AR estacional
Theta = []  # MA estacional
sigma_eps = 0.0743585003416063  # Ruido

# 1. Leer datos desde Excel --------------------------
ruta_archivo = "PrecioGasolina.xlsx"  # Cambiar ruta si es necesario
df = pd.read_excel(ruta_archivo)

def construir_serie_gasolina(df: pd.DataFrame) -> pd.Series:
    """
    A partir de un DataFrame con columnas:
        - 'Dia' (1 = 1/ene/2017)
        - 'GasolinaRegular'
    Devuelve una serie semanal (lunes) con los últimos 4 años de datos.
    """

    # 1. Convertir "Dia" → fecha real
    fecha_inicio = pd.Timestamp("2017-01-01")
    df = df.copy()
    df["fecha"] = fecha_inicio + pd.to_timedelta(df["Dia"] - 1, unit="D")

    # 2. Ordenar por fecha por si viene desordenado
    df = df.sort_values("fecha")

    # 3. Quedarnos solo con los últimos 4 años de datos
    fecha_max = df["fecha"].max()
    fecha_corte = fecha_max - pd.DateOffset(years=4)
    df = df[df["fecha"] >= fecha_corte]

    # 4. Crear serie diaria
    serie_diaria = df.set_index("fecha")["GasolinaRegular"].astype(float)

    # 5. Convertir a serie semanal (lunes)
    #    Método: tomar el promedio semanal o el valor del lunes.
    #    Aquí uso .resample('W-MON').mean()
    serie_semanal = serie_diaria.resample("W-MON").mean()

    return serie_semanal

# Construir la serie
y = construir_serie_gasolina(df)

# 2. Ajustar el modelo SARIMA con los coeficientes dados
order = (2, 0, 0)  # p, d, q
seasonal_order = (0, 0, 0, 208)  # P, D, Q, s (estacionalidad semanal con 52 semanas)

# Crear y ajustar el modelo SARIMA
modelo = sm.tsa.statespace.SARIMAX(
    y,
    order=order,
    seasonal_order=seasonal_order,
    trend="n",
    enforce_stationarity=False,
    enforce_invertibility=False,
    start_params=[*phi, *theta, *Phi, *Theta, sigma_eps]
)

# Ajustar el modelo SARIMA
res = modelo.fit(disp=False)

# Resumen del modelo
print(res.summary())

# 3. Proyección y gráfica ----------------------------
pasos_forecast = 30  # Número de semanas hacia adelante
forecast = res.get_forecast(steps=pasos_forecast)

pred_media = forecast.predicted_mean
conf_int = forecast.conf_int()  # Intervalos de confianza (por columna)

a = [1]  # Escaladores para cada gráfico
proyecciones_con_ruido = []  # Lista para almacenar las proyecciones con ruido

# Bucle para generar las gráficas con ruido
n_estaciones = 1

proyecciones_con_ruido = []
historicos_con_ruido = []

for i in range(n_estaciones):
    # Añadir ruido gaussiano a la serie histórica
    ruido_gaussiano = np.random.normal(0, sigma_eps, size=len(y))  # Ruido con media 0 y desviación sigma_eps
    y_con_ruido = y * a[i] + ruido_gaussiano  # Agregar ruido a la serie histórica

    # Guardar histórico con ruido de esta estación
    historicos_con_ruido.append(y_con_ruido)

    # Generamos ruido para las proyecciones (predicciones) también
    ruido_gaussiano2 = np.random.normal(0, sigma_eps, size=len(pred_media))
    proyeccion_con_ruido = pred_media * a[i] + ruido_gaussiano2 * 0.8 * a[i]  # Ajuste con ruido adicional

    # Guardar proyección con ruido de esta estación
    proyecciones_con_ruido.append(proyeccion_con_ruido)

    # Gráfica de la serie histórica y la proyección
    plt.figure(figsize=(12, 5))

    # Serie histórica con ruido
    plt.plot(y.index, y_con_ruido, label="Histórico (con ruido)", linewidth=2)

    # Pronóstico con ruido
    plt.plot(pred_media.index, proyeccion_con_ruido, label="Pronóstico (con ruido)", linestyle="--")

    # Banda de confianza (escalada por el factor de la estación)
    plt.fill_between(
        pred_media.index,
        conf_int.iloc[:, 0] * a[i],
        conf_int.iloc[:, 1] * a[i],
        alpha=0.2,
        label="IC 95%"
    )

    plt.title("Precio de Gasolina en México – Modelo SARIMA\nHistórico y proyección semanal")
    plt.xlabel("Fecha")
    plt.ylabel("Precio [Pesos/Litro]")
    plt.grid(True)
    plt.legend()
    plt.tight_layout()
    plt.show()

# -------------------------------
# DataFrames con estaciones como columnas
# -------------------------------

# Históricos con ruido (filas = fechas históricas, columnas = estaciones)
df_historico = pd.DataFrame(
    {f'Gasolina Regular': historicos_con_ruido[i] for i in range(n_estaciones)},
    index=y.index
)

# Proyecciones con ruido (filas = fechas de predicción, columnas = estaciones)
df_proyecciones = pd.DataFrame(
    {f'Gasolina Regular': proyecciones_con_ruido[i] for i in range(n_estaciones)},
    index=pred_media.index
)

# -------------------------------
# Exportar a Excel
# -------------------------------
with pd.ExcelWriter("PreciosGasolinaRegular.xlsx") as writer:
    df_historico.to_excel(writer, sheet_name="Historico")
    df_proyecciones.to_excel(writer, sheet_name="Proyecciones")

In [ ]:
import pandas as pd
import numpy as np
import statsmodels.api as sm
import matplotlib.pyplot as plt


phi   = [1.4499111141163346, -0.44951993542051233]  # AR no estacional
theta = []  # MA no estacional
Phi   = []  # AR estacional
Theta = []  # MA estacional
sigma_eps = 0.051066087761630435  # Ruido

# 1. Leer datos desde Excel --------------------------
ruta_archivo = "PrecioGasolinaPremium.xlsx"  # Cambiar ruta si es necesario
df = pd.read_excel(ruta_archivo)

def construir_serie_gasolina(df: pd.DataFrame) -> pd.Series:
    """
    A partir de un DataFrame con columnas:
        - 'Dia' (1 = 1/ene/2017)
        - 'GasolinaRegular'
    Devuelve una serie semanal (lunes) con los últimos 4 años de datos.
    """

    # 1. Convertir "Dia" → fecha real
    fecha_inicio = pd.Timestamp("2017-01-01")
    df = df.copy()
    df["fecha"] = fecha_inicio + pd.to_timedelta(df["Dia"] - 1, unit="D")

    # 2. Ordenar por fecha por si viene desordenado
    df = df.sort_values("fecha")

    # 3. Quedarnos solo con los últimos 4 años de datos
    fecha_max = df["fecha"].max()
    fecha_corte = fecha_max - pd.DateOffset(years=4)
    df = df[df["fecha"] >= fecha_corte]

    # 4. Crear serie diaria
    serie_diaria = df.set_index("fecha")["GasolinaPremium"].astype(float)

    # 5. Convertir a serie semanal (lunes)
    #    Método: tomar el promedio semanal o el valor del lunes.
    #    Aquí uso .resample('W-MON').mean()
    serie_semanal = serie_diaria.resample("W-MON").mean()

    return serie_semanal

# Construir la serie
y = construir_serie_gasolina(df)

# 2. Ajustar el modelo SARIMA con los coeficientes dados
order = (2, 0, 0)  # p, d, q
seasonal_order = (0, 0, 0, 208)  # P, D, Q, s (estacionalidad semanal con 52 semanas)

# Crear y ajustar el modelo SARIMA
modelo = sm.tsa.statespace.SARIMAX(
    y,
    order=order,
    seasonal_order=seasonal_order,
    trend="n",
    enforce_stationarity=False,
    enforce_invertibility=False,
    start_params=[*phi, *theta, *Phi, *Theta, sigma_eps]
)

# Ajustar el modelo SARIMA
res = modelo.fit(disp=False)

# Resumen del modelo
print(res.summary())

# 3. Proyección y gráfica ----------------------------
pasos_forecast = 30  # Número de semanas hacia adelante
forecast = res.get_forecast(steps=pasos_forecast)

pred_media = forecast.predicted_mean
conf_int = forecast.conf_int()  # Intervalos de confianza (por columna)

a = [1]  # Escaladores para cada gráfico
proyecciones_con_ruido = []  # Lista para almacenar las proyecciones con ruido

# Bucle para generar las gráficas con ruido
n_estaciones = 1

proyecciones_con_ruido = []
historicos_con_ruido = []

for i in range(n_estaciones):
    # Añadir ruido gaussiano a la serie histórica
    ruido_gaussiano = np.random.normal(0, sigma_eps, size=len(y))  # Ruido con media 0 y desviación sigma_eps
    y_con_ruido = y * a[i] + ruido_gaussiano  # Agregar ruido a la serie histórica

    # Guardar histórico con ruido de esta estación
    historicos_con_ruido.append(y_con_ruido)

    # Generamos ruido para las proyecciones (predicciones) también
    ruido_gaussiano2 = np.random.normal(0, sigma_eps, size=len(pred_media))
    proyeccion_con_ruido = pred_media * a[i] + ruido_gaussiano2 * 0.8 * a[i]  # Ajuste con ruido adicional

    # Guardar proyección con ruido de esta estación
    proyecciones_con_ruido.append(proyeccion_con_ruido)

    # Gráfica de la serie histórica y la proyección
    plt.figure(figsize=(12, 5))

    # Serie histórica con ruido
    plt.plot(y.index, y_con_ruido, label="Histórico (con ruido)", linewidth=2)

    # Pronóstico con ruido
    plt.plot(pred_media.index, proyeccion_con_ruido, label="Pronóstico (con ruido)", linestyle="--")

    # Banda de confianza (escalada por el factor de la estación)
    plt.fill_between(
        pred_media.index,
        conf_int.iloc[:, 0] * a[i],
        conf_int.iloc[:, 1] * a[i],
        alpha=0.2,
        label="IC 95%"
    )

    plt.title("Precio de Gasolina Premium en México – Modelo SARIMA\nHistórico y proyección semanal")
    plt.xlabel("Fecha")
    plt.ylabel("Precio [Pesos/Litro]")
    plt.grid(True)
    plt.legend()
    plt.tight_layout()
    plt.show()

# -------------------------------
# DataFrames con estaciones como columnas
# -------------------------------

# Históricos con ruido (filas = fechas históricas, columnas = estaciones)
df_historico = pd.DataFrame(
    {f'Gasolina Premium': historicos_con_ruido[i] for i in range(n_estaciones)},
    index=y.index
)

# Proyecciones con ruido (filas = fechas de predicción, columnas = estaciones)
df_proyecciones = pd.DataFrame(
    {f'Gasolina Premium': proyecciones_con_ruido[i] for i in range(n_estaciones)},
    index=pred_media.index
)

# -------------------------------
# Exportar a Excel
# -------------------------------
with pd.ExcelWriter("PreciosGasolinaPremium.xlsx") as writer:
    df_historico.to_excel(writer, sheet_name="Historico")
    df_proyecciones.to_excel(writer, sheet_name="Proyecciones")

In [ ]:
import pandas as pd
import numpy as np
import statsmodels.api as sm
import matplotlib.pyplot as plt


phi   = [1.3939337467391477, -0.3934140220371095]  # AR no estacional
theta = []  # MA no estacional
Phi   = []  # AR estacional
Theta = []  # MA estacional
sigma_eps = 0.05323969932263414  # Ruido

# 1. Leer datos desde Excel --------------------------
ruta_archivo = "PrecioGasolinaDiesel.xlsx"  # Cambiar ruta si es necesario
df = pd.read_excel(ruta_archivo)

def construir_serie_gasolina(df: pd.DataFrame) -> pd.Series:
    """
    A partir de un DataFrame con columnas:
        - 'Dia' (1 = 1/ene/2017)
        - 'GasolinaRegular'
    Devuelve una serie semanal (lunes) con los últimos 4 años de datos.
    """

    # 1. Convertir "Dia" → fecha real
    fecha_inicio = pd.Timestamp("2017-01-01")
    df = df.copy()
    df["fecha"] = fecha_inicio + pd.to_timedelta(df["Dia"] - 1, unit="D")

    # 2. Ordenar por fecha por si viene desordenado
    df = df.sort_values("fecha")

    # 3. Quedarnos solo con los últimos 4 años de datos
    fecha_max = df["fecha"].max()
    fecha_corte = fecha_max - pd.DateOffset(years=4)
    df = df[df["fecha"] >= fecha_corte]

    # 4. Crear serie diaria
    serie_diaria = df.set_index("fecha")["Diesel"].astype(float)

    # 5. Convertir a serie semanal (lunes)
    #    Método: tomar el promedio semanal o el valor del lunes.
    #    Aquí uso .resample('W-MON').mean()
    serie_semanal = serie_diaria.resample("W-MON").mean()

    return serie_semanal

# Construir la serie
y = construir_serie_gasolina(df)

# 2. Ajustar el modelo SARIMA con los coeficientes dados
order = (2, 0, 0)  # p, d, q
seasonal_order = (0, 0, 0, 208)  # P, D, Q, s (estacionalidad semanal con 52 semanas)

# Crear y ajustar el modelo SARIMA
modelo = sm.tsa.statespace.SARIMAX(
    y,
    order=order,
    seasonal_order=seasonal_order,
    trend="n",
    enforce_stationarity=False,
    enforce_invertibility=False,
    start_params=[*phi, *theta, *Phi, *Theta, sigma_eps]
)

# Ajustar el modelo SARIMA
res = modelo.fit(disp=False)

# Resumen del modelo
print(res.summary())

# 3. Proyección y gráfica ----------------------------
pasos_forecast = 30  # Número de semanas hacia adelante
forecast = res.get_forecast(steps=pasos_forecast)

pred_media = forecast.predicted_mean
conf_int = forecast.conf_int()  # Intervalos de confianza (por columna)

a = [1]  # Escaladores para cada gráfico
proyecciones_con_ruido = []  # Lista para almacenar las proyecciones con ruido

# Bucle para generar las gráficas con ruido
n_estaciones = 1

proyecciones_con_ruido = []
historicos_con_ruido = []

for i in range(n_estaciones):
    # Añadir ruido gaussiano a la serie histórica
    ruido_gaussiano = np.random.normal(0, sigma_eps, size=len(y))  # Ruido con media 0 y desviación sigma_eps
    y_con_ruido = y * a[i] + ruido_gaussiano  # Agregar ruido a la serie histórica

    # Guardar histórico con ruido de esta estación
    historicos_con_ruido.append(y_con_ruido)

    # Generamos ruido para las proyecciones (predicciones) también
    ruido_gaussiano2 = np.random.normal(0, sigma_eps, size=len(pred_media))
    proyeccion_con_ruido = pred_media * a[i] + ruido_gaussiano2 * 0.8 * a[i]  # Ajuste con ruido adicional

    # Guardar proyección con ruido de esta estación
    proyecciones_con_ruido.append(proyeccion_con_ruido)

    # Gráfica de la serie histórica y la proyección
    plt.figure(figsize=(12, 5))

    # Serie histórica con ruido
    plt.plot(y.index, y_con_ruido, label="Histórico (con ruido)", linewidth=2)

    # Pronóstico con ruido
    plt.plot(pred_media.index, proyeccion_con_ruido, label="Pronóstico (con ruido)", linestyle="--")

    # Banda de confianza (escalada por el factor de la estación)
    plt.fill_between(
        pred_media.index,
        conf_int.iloc[:, 0] * a[i],
        conf_int.iloc[:, 1] * a[i],
        alpha=0.2,
        label="IC 95%"
    )

    plt.title("Precio de Diesel en México – Modelo SARIMA\nHistórico y proyección semanal")
    plt.xlabel("Fecha")
    plt.ylabel("Precio [Pesos/Litro]")
    plt.grid(True)
    plt.legend()
    plt.tight_layout()
    plt.show()

# -------------------------------
# DataFrames con estaciones como columnas
# -------------------------------

# Históricos con ruido (filas = fechas históricas, columnas = estaciones)
df_historico = pd.DataFrame(
    {f'Diesel': historicos_con_ruido[i] for i in range(n_estaciones)},
    index=y.index
)

# Proyecciones con ruido (filas = fechas de predicción, columnas = estaciones)
df_proyecciones = pd.DataFrame(
    {f'Diesel': proyecciones_con_ruido[i] for i in range(n_estaciones)},
    index=pred_media.index
)

# -------------------------------
# Exportar a Excel
# -------------------------------
with pd.ExcelWriter("DieselPrecios.xlsx") as writer:
    df_historico.to_excel(writer, sheet_name="Historico")
    df_proyecciones.to_excel(writer, sheet_name="Proyecciones")